# 🌍 Week 9 — CityJSON Parsing & 3D Geospatial Data
### Geospatial Python Mastery | Module 5: 3D City Data
---
| | |
|---|---|
| **Module** | Module 5: 3D City Data |
| **Week** | 9 of 10 |
| **Duration** | 3 hours |
| **Level** | Advanced |
| **Prerequisites** | Weeks 1–8 |

> CityJSON is a JSON-based encoding of the CityGML standard for 3D city models. This week you'll master its schema, parse building objects and geometry arrays in pure Python, extract footprints and heights, validate datasets, and build practical extraction workflows for urban analytics.

### 📋 Table of Contents
| # | Section |
|---|---------|
| 1 | CityJSON Schema and Structure |
| 2 | Vertices and the Transform Model |
| 3 | CityObject Types and Hierarchy |
| 4 | Geometry Types and Boundaries |
| 5 | Extracting Building Footprints |
| 6 | Extracting Heights and Attributes |
| 7 | Validation and Error Handling |
| 8 | Reading and Writing CityJSON Files |
| 9 | Coordinate Reference Systems in CityJSON |
| 10 | Mini-Lab — Extract Building Footprints and Metadata |

### 🔣 Symbol Guide
| Symbol | Meaning |
|--------|---------|
| 💻 | Code walkthrough |
| 🎯 | Exercise |
| ✅ | Solution |
| 🔬 | Lab step |
| 🏙️ | CityJSON-specific |
| 🔷 | Runs without external files (synthetic data) |
| 📖 | Concept explanation |

*Keyboard shortcuts: `Shift+Enter` runs a cell, `A` inserts above, `B` inserts below, `Ctrl+/` toggles comments.*

## 🎯 Learning Objectives

| # | Objective |
|---|-----------|
| 1 | Explain how CityJSON encodes 3D city models using CityObjects, vertices, and transform metadata |
| 2 | Decode integer-compressed vertices with scale and translate values |
| 3 | Identify common CityJSON object types and describe parent/child hierarchy patterns |
| 4 | Interpret Solid and MultiSurface boundary nesting for LoD geometries |
| 5 | Extract 2D building footprints from LoD1 Solid geometry |
| 6 | Derive building height from both measuredHeight attributes and geometry vertices |
| 7 | Validate CityJSON structure, geometry references, and key attribute rules in Python |
| 8 | Read, write, and round-trip CityJSON files with json.load/dump and optional cjio workflows |
| 9 | Parse CRS metadata and reproject extracted geometries to WGS84 |
| 10 | Build an end-to-end extraction workflow for 3D city data analytics |

**How to use this notebook:**
- Run the notebook from top to bottom the first time so helper functions and synthetic datasets are available later.
- Treat 🔷 cells as demo-safe examples that run with synthetic data and standard Python geospatial libraries.
- Cells marked 🏙️ show CityJSON-specific workflows or optional tooling such as `cjio`.
- Pause at each 🎯 exercise and solve it before revealing the ✅ solution cell.

In [ ]:
# ── Environment detection and package installation ────────────────────────────
import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

REQUIRED = [
    'geopandas',
    'shapely',
    'pyproj',
    'pandas',
    'matplotlib',
    'numpy',
]

if IN_COLAB:
    for pkg in REQUIRED:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
    print('✅ Packages installed (Colab)')
else:
    print('ℹ️  Running locally — ensure your environment has all required packages.')
    print('   pip install', ' '.join(REQUIRED))

print('🏙️ Optional package: cjio (install separately if you want the high-level CityJSON API).')
print('   pip install cjio')
print('   Cells that rely on cjio are marked with 🏙️.')

---
## 📖 Section 1 — CityJSON Schema and Structure

CityJSON is a **JSON-based encoding of CityGML**, the OGC standard for 3D city models. It keeps the richness of CityGML, but stores it in a developer-friendly JSON structure that is much easier to parse in Python.

Key top-level keys you will see repeatedly:
- `type`, `version`, `CityObjects`, `vertices`, `transform`, `metadata`
- `transform` stores `scale = [sx, sy, sz]` and `translate = [tx, ty, tz]` so integer vertices can be decoded with `x_real = x_int * sx + tx`.
- `CityObjects` is a dictionary mapping each city object id to an object definition.
- Each city object usually includes `type`, `attributes`, `geometry`, and sometimes `children` or `parents`.
- Geometry objects define `type` (for example `Solid` or `MultiSurface`), `lod`, `boundaries`, and optional `semantics`.

Minimal valid CityJSON structure:
```json
{
  "type": "CityJSON",
  "version": "1.1",
  "transform": {
    "scale": [0.001, 0.001, 0.001],
    "translate": [84710.0, 446750.0, 0.0]
  },
  "CityObjects": {
    "building-001": {
      "type": "Building",
      "attributes": {"measuredHeight": 9.5, "yearBuilt": 1960},
      "geometry": [...]
    }
  },
  "vertices": [[84710, 446750, 0], [85000, 446750, 0], ...]
}
```

In [ ]:
# 💻 1.2  Build a minimal synthetic CityJSON dictionary (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import json

# Minimal synthetic CityJSON with 2 buildings
cityjson = {
    "type": "CityJSON",
    "version": "1.1",
    "transform": {
        "scale": [0.001, 0.001, 0.001],
        "translate": [84710.0, 446750.0, 0.0]
    },
    "metadata": {
        "referenceSystem": "https://www.opengis.net/def/crs/EPSG/0/28992",
        "geographicalExtent": [84710.0, 446750.0, 0.0, 85500.0, 447500.0, 15.0]
    },
    "CityObjects": {
        "building-001": {
            "type": "Building",
            "attributes": {"measuredHeight": 9.5, "yearBuilt": 1960, "function": "residential"},
            "geometry": []
        },
        "building-002": {
            "type": "Building",
            "attributes": {"measuredHeight": 24.3, "yearBuilt": 1998, "function": "office"},
            "geometry": []
        },
        "road-001": {
            "type": "Road",
            "attributes": {"name": "Stationsplein", "width": 12.0},
            "geometry": []
        }
    },
    "vertices": []
}

print(f"CityJSON version : {cityjson['version']}")
print(f"CRS              : {cityjson['metadata']['referenceSystem']}")
print(f"City objects     : {len(cityjson['CityObjects'])}")
for oid, obj in cityjson['CityObjects'].items():
    print(f"  {oid:<15} type={obj['type']}")
print(f"Transform scale  : {cityjson['transform']['scale']}")
print(f"Transform origin : {cityjson['transform']['translate']}")

In [ ]:
# 💻 1.3  Summarise object types and metadata (🔷)
# ─────────────────────────────────────────────────────────────────────────────
from collections import Counter

# Count city object types
type_counts = Counter(obj['type'] for obj in cityjson['CityObjects'].values())
print("Object type counts:")
for t, n in type_counts.most_common():
    print(f"  {t:<20} : {n}")

# Extract all attributes across all objects
all_attrs = {}
for oid, obj in cityjson['CityObjects'].items():
    for k, v in obj.get('attributes', {}).items():
        all_attrs.setdefault(k, []).append(v)

print("
Attribute summary:")
for attr, vals in all_attrs.items():
    print(f"  {attr:<20} : {vals}")

print(f"
Geographical extent: {cityjson['metadata'].get('geographicalExtent','N/A')}")

In [ ]:
# 💻 1.4  Inspect key schema components (🔷)
# ─────────────────────────────────────────────────────────────────────────────
required_keys = ['type', 'version', 'CityObjects', 'vertices', 'transform', 'metadata']
print("Top-level key presence:")
for key in required_keys:
    print(f"  {key:<12} -> {key in cityjson}")

building_like = [oid for oid, obj in cityjson['CityObjects'].items() if obj['type'].startswith('Building')]
non_building = [oid for oid, obj in cityjson['CityObjects'].items() if not obj['type'].startswith('Building')]
print(f"
Building-like objects : {building_like}")
print(f"Other object ids      : {non_building}")
print(f"Metadata keys         : {sorted(cityjson['metadata'].keys())}")

In [ ]:
# 💻 1.5  Flatten CityObjects into rows (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd

rows = []
for oid, obj in cityjson['CityObjects'].items():
    row = {'id': oid, 'type': obj['type'], 'n_geometry': len(obj.get('geometry', []))}
    row.update(obj.get('attributes', {}))
    rows.append(row)

schema_df = pd.DataFrame(rows).fillna("—")
print(schema_df.to_string(index=False))

### 📖 Section 1 takeaway

- CityJSON separates **object metadata** (`CityObjects`) from **coordinate storage** (`vertices` + `transform`).
- Before touching geometry, always inspect the schema version, CRS metadata, and the mix of object types present.

---
## 📖 Section 2 — Vertices and the Transform Model

CityJSON compresses geometry by storing all vertices as integer triplets `[ix, iy, iz]` and placing the real-world scaling in a `transform` object.

- All vertices are stored as integers `[ix, iy, iz]`.
- Real coordinates are recovered with `x = ix * scale[0] + translate[0]`, and the same pattern applies to `y` and `z`.
- This often reduces JSON file size by roughly **50% compared with raw float storage**.
- `transform.scale` is often `[0.001, 0.001, 0.001]`, which gives millimetre precision in a metre-based CRS.

Worked example:
- Vertex `[500, 290, 8000]`
- Scale `[0.001, 0.001, 0.001]`
- Translate `[84710.0, 446750.0, 0.0]`
- Real coordinates = `(84710.500, 446750.290, 8.000)`

In [ ]:
# 💻 2.2  Decode integer-compressed vertices (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np

def decode_vertices(vertices: list, transform: dict) -> np.ndarray:
    scale = np.array(transform['scale'])
    translate = np.array(transform['translate'])
    verts = np.array(vertices, dtype=np.float64)
    return verts * scale + translate

# Build a realistic synthetic vertex list (integer-compressed)
# RD New coords for a small Amsterdam block
raw_vertices = [
    [500,  290, 0],      # ground-floor ring
    [790,  290, 0],
    [790,  710, 0],
    [500,  710, 0],
    [500,  290, 9500],   # roof-ring
    [790,  290, 9500],
    [790,  710, 9500],
    [500,  710, 9500],
]

transform = {"scale": [0.001, 0.001, 0.001], "translate": [84710.0, 446750.0, 0.0]}
real = decode_vertices(raw_vertices, transform)

print("idx   ix    iy    iz  |  x (m)         y (m)        z (m)")
print('-' * 65)
for i, (raw, decoded) in enumerate(zip(raw_vertices, real)):
    print(f"  {i}   {raw[0]:5d} {raw[1]:5d} {raw[2]:5d}  |  {decoded[0]:12.3f}  {decoded[1]:12.3f}  {decoded[2]:8.3f}")

In [ ]:
# 💻 2.3  Compute bounding box and centroid (🔷)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nBounding box (decoded vertices):")
print(f"  X : {real[:,0].min():.3f} → {real[:,0].max():.3f}  (width  = {np.ptp(real[:,0]):.3f} m)")
print(f"  Y : {real[:,1].min():.3f} → {real[:,1].max():.3f}  (depth  = {np.ptp(real[:,1]):.3f} m)")
print(f"  Z : {real[:,2].min():.3f} → {real[:,2].max():.3f}  (height = {np.ptp(real[:,2]):.3f} m)")

# Centroid
centroid = real.mean(axis=0)
print(f"\nCentroid: X={centroid[0]:.3f}  Y={centroid[1]:.3f}  Z={centroid[2]:.3f}")

In [ ]:
# 💻 2.4  Worked transform example (🔷)
# ─────────────────────────────────────────────────────────────────────────────
example_vertex = [500, 290, 8000]
example_real = decode_vertices([example_vertex], transform)[0]
print(f"Integer vertex : {example_vertex}")
print(f"Scale          : {transform['scale']}")
print(f"Translate      : {transform['translate']}")
print(f"Real coords    : ({example_real[0]:.3f}, {example_real[1]:.3f}, {example_real[2]:.3f})")

In [ ]:
# 💻 2.5  Round-trip precision check (🔷)
# ─────────────────────────────────────────────────────────────────────────────
def encode_vertex(x, y, z, transform):
    sx, sy, sz = transform['scale']
    tx, ty, tz = transform['translate']
    return [int(round((x - tx) / sx)), int(round((y - ty) / sy)), int(round((z - tz) / sz))]

round_trip_rows = []
for decoded in real[:3]:
    encoded = encode_vertex(decoded[0], decoded[1], decoded[2], transform)
    decoded_again = decode_vertices([encoded], transform)[0]
    round_trip_rows.append({
        "original": tuple(round(v, 3) for v in decoded),
        "encoded": encoded,
        "decoded_again": tuple(round(v, 3) for v in decoded_again),
    })
print(pd.DataFrame(round_trip_rows).to_string(index=False))

### 🎯 Exercise 1 — Decode vertices with a new transform

**Task:** Given a new transform with scale `[0.0001, 0.0001, 0.001]` and translate `[100000.0, 400000.0, 0.0]`, decode the vertex list `[[12000, 8500, 0], [14500, 8500, 0], [14500, 11000, 0], [12000, 11000, 0]]`. Print the real coordinates and the footprint area in m².

**Steps:**
1. Create the new transform and vertex list.
2. Decode the vertices with `decode_vertices()`.
3. Build a polygon from the XY coordinates and print its area.

**Hint:**

```python
decoded = decode_vertices(vertices, new_transform)
```

In [ ]:
# 🎯 Exercise 1 — your code here ────────────────────────────────
# 1. Create the new transform and raw vertex list.
# 2. Decode the vertices.
# 3. Create a polygon from XY coordinates and print the area in m².

In [ ]:
# ✅ Exercise 1 — Solution ──────────────────────────────────────
from shapely.geometry import Polygon

new_transform = {'scale': [0.0001, 0.0001, 0.001], 'translate': [100000.0, 400000.0, 0.0]}
vertices = [[12000, 8500, 0], [14500, 8500, 0], [14500, 11000, 0], [12000, 11000, 0]]
decoded = decode_vertices(vertices, new_transform)
print(decoded)
poly = Polygon([(x, y) for x, y, _ in decoded])
print(f"Footprint area: {poly.area:.2f} m²")

### 📖 Section 2 takeaway

- Never interpret CityJSON vertices as real coordinates until you apply `scale` and `translate`.
- Once decoded, you can compute normal geometry metrics such as bounding boxes, centroids, areas, and heights.

---
## 📖 Section 3 — CityObject Types and Hierarchy

| Type | Description | Typical attributes |
|------|-------------|-------------------|
| Building | Building shell | measuredHeight, yearBuilt, function, roofType |
| BuildingPart | Part of a building | same as Building |
| BuildingInstallation | Balcony, chimney, etc | — |
| Road | Road surface | name, width, surface |
| WaterBody | River, lake | waterBodyClass |
| LandUse | Zoning area | class, function, usage |
| CityFurniture | Street furniture | class, function |
| Bridge | Bridge structure | — |
| Tunnel | Underground tunnel | — |

Hierarchy matters: a `Building` can have `children` that are `BuildingPart` objects. The parent often acts as the aggregate object, while the detailed geometry and attributes may sit on the parts.

In [ ]:
# 💻 3.2  Build a Building + BuildingPart hierarchy (🔷)
# ─────────────────────────────────────────────────────────────────────────────
cityjson2 = {
    "type": "CityJSON",
    "version": "1.1",
    "transform": {"scale": [0.001, 0.001, 0.001], "translate": [84710.0, 446750.0, 0.0]},
    "CityObjects": {
        "B001": {
            "type": "Building",
            "attributes": {"measuredHeight": 12.0, "yearBuilt": 1975, "function": "residential", "storeysAboveGround": 4},
            "children": ["B001-part1", "B001-part2"],
            "geometry": []
        },
        "B001-part1": {
            "type": "BuildingPart",
            "attributes": {"measuredHeight": 8.0},
            "parents": ["B001"],
            "geometry": []
        },
        "B001-part2": {
            "type": "BuildingPart",
            "attributes": {"measuredHeight": 12.0},
            "parents": ["B001"],
            "geometry": []
        },
        "B002": {
            "type": "Building",
            "attributes": {"measuredHeight": 35.0, "yearBuilt": 2005, "function": "office", "storeysAboveGround": 10},
            "geometry": []
        },
        "R001": {"type": "Road", "attributes": {"name": "Damrak", "width": 18.0}, "geometry": []},
        "W001": {"type": "WaterBody", "attributes": {"waterBodyClass": "canal"}, "geometry": []},
    },
    "vertices": []
}

# Walk the hierarchy
print("City object hierarchy:")
for oid, obj in cityjson2['CityObjects'].items():
    children = obj.get('children', [])
    parents  = obj.get('parents', [])
    suffix = f" → children: {children}" if children else (f" ↑ parent: {parents[0]}" if parents else "")
    print(f"  {oid:<14} [{obj['type']:<20}]{suffix}")

In [ ]:
# 💻 3.3  Extract object attributes to a DataFrame (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd

def extract_objects(cj: dict, obj_types=None) -> pd.DataFrame:
    rows = []
    for oid, obj in cj['CityObjects'].items():
        if obj_types and obj['type'] not in obj_types:
            continue
        row = {'id': oid, 'type': obj['type']}
        row.update(obj.get('attributes', {}))
        row['n_children'] = len(obj.get('children', []))
        row['n_parents']  = len(obj.get('parents', []))
        rows.append(row)
    return pd.DataFrame(rows)

buildings = extract_objects(cityjson2, ['Building', 'BuildingPart'])
print(buildings.to_string(index=False))
print(f"\nAll objects:")
print(extract_objects(cityjson2).to_string(index=False))

In [ ]:
# 💻 3.4  Summarise hierarchy metrics (🔷)
# ─────────────────────────────────────────────────────────────────────────────
hierarchy_rows = []
for oid, obj in cityjson2['CityObjects'].items():
    hierarchy_rows.append({
        "id": oid,
        "type": obj["type"],
        "children": len(obj.get("children", [])),
        "parents": len(obj.get("parents", [])),
    })
hierarchy_df = pd.DataFrame(hierarchy_rows)
print(hierarchy_df.to_string(index=False))
print("
Counts by type:")
print(hierarchy_df.groupby("type")[["children", "parents"]].sum().to_string())

In [ ]:
# 💻 3.5  Trace parent and child links (🔷)
# ─────────────────────────────────────────────────────────────────────────────
def describe_lineage(cj: dict, oid: str) -> dict:
    obj = cj['CityObjects'][oid]
    return {
        "id": oid,
        "type": obj["type"],
        "children": obj.get("children", []),
        "parents": obj.get("parents", []),
    }

print(describe_lineage(cityjson2, "B001"))
print(describe_lineage(cityjson2, "B001-part1"))

### 🎯 Exercise 2 — Return a Building with its parts

**Task:** From `cityjson2`, write a function `get_building_with_parts(cj, building_id)` that returns the parent Building attributes merged with a list of its BuildingPart attributes.

**Steps:**
1. Fetch the parent object from `CityObjects`.
2. Loop through its `children` and collect each child object and attributes.
3. Return a structured dictionary that includes parent attributes and a `parts` list.

**Hint:**

```python
parent = cj['CityObjects'][building_id]
```

In [ ]:
# 🎯 Exercise 2 — your code here ────────────────────────────────
# 1. Get the parent Building object.
# 2. Loop through child ids and collect their attributes.
# 3. Return a dictionary with the parent info and a list of parts.

In [ ]:
# ✅ Exercise 2 — Solution ──────────────────────────────────────
def get_building_with_parts(cj, building_id):
    parent = cj['CityObjects'][building_id]
    result = {
        "id": building_id,
        "type": parent["type"],
        "attributes": dict(parent.get("attributes", {})),
        "parts": [],
    }
    for child_id in parent.get("children", []):
        child = cj['CityObjects'][child_id]
        result["parts"].append({
            "id": child_id,
            "type": child["type"],
            "attributes": dict(child.get("attributes", {})),
        })
    return result

print(get_building_with_parts(cityjson2, "B001"))

### 📖 Section 3 takeaway

- `CityObjects` is more than a flat table: object ids, types, and hierarchy links all matter.
- When parsing buildings, check whether geometry sits on the parent, the parts, or both.

---
## 📖 Section 4 — Geometry Types and Boundaries

| Geometry type | Description | Boundary nesting |
|--------------|-------------|-----------------|
| MultiPoint | Set of points | `[[v0], [v1], ...]` |
| MultiLineString | Set of line strings | `[[[v0,v1,v2]], ...]` |
| MultiSurface | Set of polygons (LoD1/2 surfaces) | `[[[ring]], [[ring,hole]]]` |
| CompositeSurface | Surfaces forming a closed shell | same as MultiSurface |
| Solid | Closed volume (exterior + interior shells) | `[[exterior_shell], [interior_shell]]` |
| MultiSolid | Set of Solids | nests one level deeper |

Each geometry also carries a `lod` (Level of Detail):
- LoD0: footprint only (2D)
- LoD1: extruded block (flat roof)
- LoD2: detailed roof surfaces
- LoD3: full architectural detail

In [ ]:
# 💻 4.2  Build a LoD1 Solid geometry (🔷)
# ─────────────────────────────────────────────────────────────────────────────
# LoD1 solid = box building: 8 vertices, 6 faces
# Vertices already in cityjson['vertices'] list at indices 0–7
# Face ring order: counter-clockwise when viewed from outside

lod1_solid_geometry = {
    "type": "Solid",
    "lod": "1",
    "boundaries": [
        [  # exterior shell
            [[0, 3, 2, 1]],  # bottom face (ground)
            [[4, 5, 6, 7]],  # top face (roof)
            [[0, 1, 5, 4]],  # front face
            [[1, 2, 6, 5]],  # right face
            [[2, 3, 7, 6]],  # back face
            [[3, 0, 4, 7]],  # left face
        ]
    ]
}

print("LoD1 Solid geometry:")
print(f"  type      : {lod1_solid_geometry['type']}")
print(f"  lod       : {lod1_solid_geometry['lod']}")
n_shells = len(lod1_solid_geometry['boundaries'])
n_faces  = sum(len(shell) for shell in lod1_solid_geometry['boundaries'])
n_rings  = sum(len(face) for shell in lod1_solid_geometry['boundaries'] for face in shell)
print(f"  shells    : {n_shells}")
print(f"  faces     : {n_faces}")
print(f"  rings     : {n_rings}")

# Show each face with its vertex indices
print("\n  Face breakdown:")
for fi, face in enumerate(lod1_solid_geometry['boundaries'][0]):
    ring = face[0]  # outer ring
    print(f"    face {fi}: vertices {ring}")

In [ ]:
# 💻 4.3  Collect vertex indices recursively (🔷)
# ─────────────────────────────────────────────────────────────────────────────
def collect_vertex_indices(boundaries, depth=0) -> list:
    indices = []
    for item in boundaries:
        if not item:
            continue
        if isinstance(item[0], int):
            # This is a ring: list of vertex indices
            indices.extend(item)
        else:
            indices.extend(collect_vertex_indices(item, depth+1))
    return indices

all_verts_used = collect_vertex_indices(lod1_solid_geometry['boundaries'])
unique_verts   = sorted(set(all_verts_used))
print(f"All vertex index references : {all_verts_used}")
print(f"Unique vertices used        : {unique_verts}")
print(f"Total index references      : {len(all_verts_used)}")

# Decode the vertices using our transform decoder
full_vertices = raw_vertices  # from Section 2
real_verts = decode_vertices(full_vertices, transform)
used_real = real_verts[unique_verts]
print(f"\nReal-world coordinates of used vertices (RD New):")
for vi, rv in zip(unique_verts, used_real):
    print(f"  v{vi}: ({rv[0]:.3f}, {rv[1]:.3f}, {rv[2]:.3f})")

In [ ]:
# 💻 4.4  Rank faces by mean Z value (🔷)
# ─────────────────────────────────────────────────────────────────────────────
face_rows = []
for fi, face in enumerate(lod1_solid_geometry['boundaries'][0]):
    ring = face[0]
    coords = real_verts[ring]
    face_rows.append({
        "face": fi,
        "ring": ring,
        "mean_z": round(float(coords[:, 2].mean()), 3),
    })
face_df = pd.DataFrame(face_rows).sort_values("mean_z")
print(face_df.to_string(index=False))

In [ ]:
# 💻 4.5  Preview a semantics block (🔷)
# ─────────────────────────────────────────────────────────────────────────────
lod1_semantics = {
    "surfaces": [
        {"type": "GroundSurface"},
        {"type": "RoofSurface"},
        {"type": "WallSurface"},
    ],
    "values": [0, 1, 2, 2, 2, 2],
}
print(json.dumps(lod1_semantics, indent=2))

### 📖 Section 4 takeaway

- The hardest part of CityJSON parsing is usually understanding how deeply nested the `boundaries` array is for each geometry type.
- Recursive traversal is the safest general-purpose approach for collecting vertex references from geometry objects.

---
## 📖 Section 5 — Extracting Building Footprints

Footprint extraction depends on geometry type and LoD:
- LoD0 geometry: a `MultiSurface` on the ground plane directly gives a footprint polygon.
- LoD1 `Solid`: take the bottom face of the exterior shell and project its vertices to 2D.
- LoD2+: find faces with a downward normal, or more practically, find faces at minimum Z.

**Best practice:** sort candidate faces by mean Z and take the lowest face ring as the footprint.

In [ ]:
# 💻 5.2  Extract a footprint polygon from LoD1 Solid geometry (🔷)
# ─────────────────────────────────────────────────────────────────────────────
from shapely.geometry import Polygon
import numpy as np

def extract_footprint(geometry: dict, vertices_real: np.ndarray) -> Polygon:
    if geometry['type'] not in ('Solid', 'MultiSurface', 'CompositeSurface'):
        return None

    # For Solid: iterate all faces, find the one with the lowest mean Z
    shells = geometry['boundaries']
    shell  = shells[0]  # exterior shell

    best_face  = None
    best_z     = float('inf')

    for face in shell:
        outer_ring = face[0]  # first ring = outer boundary
        coords     = vertices_real[outer_ring]
        mean_z     = coords[:, 2].mean()
        if mean_z < best_z:
            best_z    = mean_z
            best_face = outer_ring

    if best_face is None:
        return None

    # Project to 2D (drop Z)
    xy_coords = [(vertices_real[vi, 0], vertices_real[vi, 1]) for vi in best_face]
    return Polygon(xy_coords)


footprint = extract_footprint(lod1_solid_geometry, real_verts)
print(f"Footprint type   : {footprint.geom_type}")
print(f"Footprint valid  : {footprint.is_valid}")
print(f"Footprint area   : {footprint.area:.2f} m²")
print(f"Footprint bounds : {tuple(round(v,3) for v in footprint.bounds)}")
print(f"Footprint WKT    : {footprint.wkt}")

In [ ]:
# 💻 5.3  Batch extract footprints to a GeoDataFrame (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import geopandas as gpd
import pandas as pd

# Build a synthetic CityJSON with 4 buildings with full geometry
def make_box_building(cx, cy, width, depth, height, transform):
    sx, sy, sz = transform['scale']
    tx, ty, tz = transform['translate']

    def to_int(x, y, z):
        return [int((x - tx) / sx), int((y - ty) / sy), int((z - tz) / sz)]

    base_verts = [
        to_int(cx,       cy,       0),
        to_int(cx+width, cy,       0),
        to_int(cx+width, cy+depth, 0),
        to_int(cx,       cy+depth, 0),
        to_int(cx,       cy,       height),
        to_int(cx+width, cy,       height),
        to_int(cx+width, cy+depth, height),
        to_int(cx,       cy+depth, height),
    ]
    geom = {
        "type": "Solid",
        "lod": "1",
        "boundaries": [[
            [[0,3,2,1]], [[4,5,6,7]],
            [[0,1,5,4]], [[1,2,6,5]],
            [[2,3,7,6]], [[3,0,4,7]],
        ]]
    }
    return base_verts, geom

tf = {"scale": [0.001,0.001,0.001], "translate": [84710.0, 446750.0, 0.0]}

buildings_spec = [
    ("B_A", 84720.0, 446760.0, 15.0, 12.0, 9.5,  1960, "residential"),
    ("B_B", 84740.0, 446760.0, 20.0, 15.0, 24.3, 1998, "office"),
    ("B_C", 84720.0, 446790.0, 10.0, 10.0, 7.0,  1930, "residential"),
    ("B_D", 84760.0, 446770.0, 25.0, 20.0, 42.0, 2010, "commercial"),
]

all_vertices = []
city_objects = {}
for bid, cx, cy, w, d, h, yr, func in buildings_spec:
    v_offset = len(all_vertices)
    verts, geom = make_box_building(cx, cy, w, d, h, tf)
    all_vertices.extend(verts)
    # Reindex geometry boundaries
    def reindex(boundaries, offset):
        def _ri(item):
            if isinstance(item[0], int):
                return [v + offset for v in item]
            return [_ri(sub) for sub in item]
        return [_ri(shell) for shell in boundaries]
    geom['boundaries'] = reindex(geom['boundaries'], v_offset)
    city_objects[bid] = {
        "type": "Building",
        "attributes": {"measuredHeight": h, "yearBuilt": yr, "function": func},
        "geometry": [geom]
    }

cj_full = {
    "type": "CityJSON",
    "version": "1.1",
    "transform": tf,
    "metadata": {
        "referenceSystem": "https://www.opengis.net/def/crs/EPSG/0/28992",
        "geographicalExtent": [84720.0, 446760.0, 0.0, 84785.0, 446810.0, 42.0],
    },
    "CityObjects": city_objects,
    "vertices": all_vertices,
}

# Extract all footprints
real_all = decode_vertices(cj_full['vertices'], cj_full['transform'])
rows = []
for bid, obj in cj_full['CityObjects'].items():
    for geom in obj['geometry']:
        fp = extract_footprint(geom, real_all)
        if fp is not None:
            row = {'id': bid, 'geometry': fp}
            row.update(obj['attributes'])
            rows.append(row)

gdf = gpd.GeoDataFrame(rows, crs='EPSG:28992')
print(gdf[['id','measuredHeight','yearBuilt','function']].to_string(index=False))
print(f"\nTotal footprints extracted: {len(gdf)}")
print(f"Total footprint area      : {gdf.geometry.area.sum():.2f} m²")

In [ ]:
# 💻 5.4  Preview candidate faces for footprint selection (🔷)
# ─────────────────────────────────────────────────────────────────────────────
candidate_faces = []
for face in lod1_solid_geometry['boundaries'][0]:
    ring = face[0]
    coords = real_verts[ring]
    candidate_faces.append({
        "ring": ring,
        "mean_z": round(float(coords[:, 2].mean()), 3),
        "area_proxy": round(float(Polygon([(x, y) for x, y, _ in coords]).area), 3),
    })
print(pd.DataFrame(candidate_faces).sort_values("mean_z").to_string(index=False))

In [ ]:
# 💻 5.5  Summarise footprint area by function (🔷)
# ─────────────────────────────────────────────────────────────────────────────
gdf['area_m2'] = gdf.geometry.area.round(2)
print(gdf[["id", "function", "area_m2"]].to_string(index=False))
print("
Area by function:")
print(gdf.groupby("function")["area_m2"].sum().round(2).to_string())

### 🎯 Exercise 3 — Add area and FAR-style proxy columns

**Task:** Add an `area_m2` column to the GeoDataFrame containing each building's footprint area, then compute an estimated storey count and a simple FAR-style proxy using `measuredHeight / 3.0`.

**Steps:**
1. Compute `area_m2` from the geometry column.
2. Estimate floors with `round(measuredHeight / 3)`.
3. Create a `far_proxy` column from `measuredHeight / 3.0` and print the result.

**Hint:**

```python
gdf['area_m2'] = gdf.geometry.area
```

In [ ]:
# 🎯 Exercise 3 — your code here ────────────────────────────────
# 1. Add an area_m2 column.
# 2. Estimate the number of storeys from measuredHeight.
# 3. Compute a FAR-style proxy and print selected columns.

In [ ]:
# ✅ Exercise 3 — Solution ──────────────────────────────────────
gdf['area_m2'] = gdf.geometry.area.round(2)
gdf['est_floors'] = (gdf['measuredHeight'] / 3.0).round().astype(int)
gdf['far_proxy'] = (gdf['measuredHeight'] / 3.0).round(2)
print(gdf[["id", "area_m2", "measuredHeight", "est_floors", "far_proxy"]].to_string(index=False))

### 📖 Section 5 takeaway

- LoD1 footprint extraction is usually a **lowest-face selection** problem.
- Once you have a Shapely polygon, CityJSON building shells become easy to analyse with GeoPandas.

---
## 📖 Section 6 — Extracting Heights and Attributes

There are several ways to derive building height from CityJSON data:
- `attributes.measuredHeight`: direct attribute and usually the most reliable value.
- Geometry-derived height: `max(z) - min(z)` across all vertices referenced by the object.
- LoD2-specific logic: compare roof vertex Z values to ground vertex Z values.
- `storeysAboveGround`: useful when present, but it describes floors rather than metric height.

**Best practice:** prefer `measuredHeight` when available, and use geometry-derived height as a validation check or fallback.

In [ ]:
# 💻 6.2  Compare attribute height with geometry-derived height (🔷)
# ─────────────────────────────────────────────────────────────────────────────
def get_height_from_geometry(obj: dict, vertices_real: np.ndarray) -> float:
    all_z = []
    for geom in obj.get('geometry', []):
        indices = collect_vertex_indices(geom['boundaries'])
        zvals   = vertices_real[indices, 2]
        all_z.extend(zvals.tolist())
    if not all_z:
        return None
    return max(all_z) - min(all_z)

print(f"{'ID':<8} {'attr_h':>8} {'geom_h':>8} {'match':>6}  function")
print('-' * 50)
for bid, obj in cj_full['CityObjects'].items():
    attr_h = obj['attributes'].get('measuredHeight')
    geom_h = get_height_from_geometry(obj, real_all)
    match  = abs(attr_h - geom_h) < 0.1 if (attr_h is not None and geom_h is not None) else "N/A"
    attr_text = f"{attr_h:.1f}" if attr_h is not None else "N/A"
    geom_text = f"{geom_h:.1f}" if geom_h is not None else "N/A"
    print(f"{bid:<8}  {attr_text:>7}  {geom_text:>7}  {str(match):>6}  {obj['attributes']['function']}")

In [ ]:
# 💻 6.3  Build a summary statistics table (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd

stats = []
for bid, obj in cj_full['CityObjects'].items():
    attrs = obj['attributes']
    geom_h = get_height_from_geometry(obj, real_all)
    fp = next((extract_footprint(g, real_all) for g in obj["geometry"]), None)
    stats.append({
        "id"            : bid,
        "type"          : obj["type"],
        "function"      : attrs.get("function","?"),
        "height_m"      : attrs.get("measuredHeight", geom_h),
        "year_built"    : attrs.get("yearBuilt"),
        "footprint_m2"  : round(fp.area, 2) if fp else None,
        "volume_m3"     : round(fp.area * attrs.get("measuredHeight", 0), 2) if fp else None,
    })

df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))
print(f"\nMean height  : {df_stats['height_m'].mean():.1f} m")
print(f"Total volume : {df_stats['volume_m3'].sum():.0f} m³")

In [ ]:
# 💻 6.4  Estimate floors from height (🔷)
# ─────────────────────────────────────────────────────────────────────────────
df_stats['est_storeys'] = (df_stats['height_m'] / 3.0).round().astype(int)
print(df_stats[["id", "function", "height_m", "est_storeys"]].to_string(index=False))

In [ ]:
# 💻 6.5  Rank buildings by estimated volume (🔷)
# ─────────────────────────────────────────────────────────────────────────────
ranked = df_stats.sort_values(["volume_m3", "height_m"], ascending=[False, False]).reset_index(drop=True)
ranked.index = ranked.index + 1
print(ranked[["id", "function", "height_m", "footprint_m2", "volume_m3"]].to_string())

### 📖 Section 6 takeaway

- Height extraction is a good integrity check: geometry and attributes should broadly agree.
- Derived metrics such as storeys, volume, and density become straightforward once heights and footprints are available.

---
## 📖 Section 7 — Validation and Error Handling

| Issue | Description | Check |
|-------|-------------|-------|
| Missing transform | No transform key | `'transform' in cj` |
| Empty geometry | CityObject with no geometry | `len(obj["geometry"]) == 0` |
| Invalid vertex index | Boundary references out-of-range index | `max(indices) < len(vertices)` |
| Non-planar face | Face vertices not coplanar | normal vector check |
| Zero-area polygon | Degenerate face | `polygon.area == 0` |
| Duplicate vertices | Same vertex stored twice | hash comparison |

Validation keeps your parser safe: check schema completeness, geometry references, and attribute quality before computing analytics or exporting derived products.

In [ ]:
# 💻 7.2  Write a basic validate_cityjson() function (🔷)
# ─────────────────────────────────────────────────────────────────────────────
def validate_cityjson(cj: dict) -> dict:
    errors   = []
    warnings = []
    n_verts  = len(cj.get('vertices', []))

    # Check top-level keys
    for key in ('type', 'version', 'CityObjects', 'vertices'):
        if key not in cj:
            errors.append(f"Missing required key: {key!r}")

    if cj.get('type') != 'CityJSON':
        errors.append(f"type should be 'CityJSON', got {cj.get('type')!r}")

    if 'transform' not in cj:
        warnings.append("No 'transform' key — vertices assumed to be real-world coordinates")

    # Validate each city object
    for oid, obj in cj.get('CityObjects', {}).items():
        if 'type' not in obj:
            errors.append(f"{oid}: missing 'type'")
        if not obj.get('geometry'):
            warnings.append(f"{oid}: no geometry defined")
        for gi, geom in enumerate(obj.get('geometry', [])):
            try:
                indices = collect_vertex_indices(geom['boundaries'])
                if indices and max(indices) >= n_verts:
                    errors.append(f"{oid}/geom[{gi}]: vertex index {max(indices)} out of range (n_verts={n_verts})")
            except Exception as e:
                errors.append(f"{oid}/geom[{gi}]: boundary parse error: {e}")

    return {
        'valid'    : len(errors) == 0,
        'errors'   : errors,
        'warnings' : warnings,
        'n_objects': len(cj.get('CityObjects', {})),
        'n_vertices': n_verts,
    }

result = validate_cityjson(cj_full)
print(f"Valid    : {result['valid']}")
print(f"Errors   : {result['errors'] or 'none'}")
print(f"Warnings : {result['warnings'] or 'none'}")
print(f"Objects  : {result['n_objects']}")
print(f"Vertices : {result['n_vertices']}")

In [ ]:
# 💻 7.3  Test validation with a broken dataset (🔷)
# ─────────────────────────────────────────────────────────────────────────────
# Introduce deliberate errors
broken = {
    "type": "CityJSON",
    "version": "1.1",
    "CityObjects": {
        "bad-obj": {
            "type": "Building",
            "geometry": [{
                "type": "Solid",
                "lod": "1",
                "boundaries": [[[[0, 999, 1]]]]  # vertex 999 out of range
            }]
        },
        "no-geom-obj": {
            "type": "Road",
            "geometry": []   # no geometry
        }
    },
    "vertices": [[0,0,0],[100,0,0],[100,100,0]]
}

broken_result = validate_cityjson(broken)
print(f"Valid    : {broken_result['valid']}")
for e in broken_result['errors']:
    print(f"  ERROR  : {e}")
for w in broken_result['warnings']:
    print(f"  WARN   : {w}")

In [ ]:
# 💻 7.4  Scan for duplicate vertices (🔷)
# ─────────────────────────────────────────────────────────────────────────────
from collections import Counter

vertex_counter = Counter(tuple(v) for v in cj_full["vertices"])
duplicates = [vertex for vertex, count in vertex_counter.items() if count > 1]
print(f"Duplicate vertex count: {len(duplicates)}")
if duplicates:
    print(duplicates[:5])
else:
    print("No duplicate vertices found in cj_full")

In [ ]:
# 💻 7.5  Summarise validation reports (🔷)
# ─────────────────────────────────────────────────────────────────────────────
validation_rows = [
    {"dataset": "cj_full", **validate_cityjson(cj_full)},
    {"dataset": "broken", **validate_cityjson(broken)},
]
validation_df = pd.DataFrame(validation_rows)
print(validation_df[["dataset", "valid", "n_objects", "n_vertices"]].to_string(index=False))
print("
Detailed warnings/errors:")
for row in validation_rows:
    print(f"- {row['dataset']}: errors={len(row['errors'])}, warnings={len(row['warnings'])}")

### 🎯 Exercise 4 — Validate measuredHeight values

**Task:** Extend `validate_cityjson()` to also check that each Building's `measuredHeight` attribute (if present) is a positive number. Test it with a building that has `measuredHeight: -5`.

**Steps:**
1. Copy `validate_cityjson()` to a new function.
2. Add a rule for Building objects with non-positive `measuredHeight`.
3. Create a small test dataset and print the errors.

**Hint:**

```python
if obj['type'] == 'Building' and 'measuredHeight' in attrs:
```

In [ ]:
# 🎯 Exercise 4 — your code here ────────────────────────────────
# 1. Create an extended validator function.
# 2. Add a positive measuredHeight check for Building objects.
# 3. Test it on a building with measuredHeight = -5.

In [ ]:
# ✅ Exercise 4 — Solution ──────────────────────────────────────
def validate_cityjson_with_height(cj: dict) -> dict:
    result = validate_cityjson(cj)
    errors = list(result["errors"])
    for oid, obj in cj.get('CityObjects', {}).items():
        attrs = obj.get("attributes", {})
        if obj.get("type") == "Building" and "measuredHeight" in attrs:
            try:
                if float(attrs["measuredHeight"]) <= 0:
                    errors.append(f"{oid}: measuredHeight must be positive")
            except Exception:
                errors.append(f"{oid}: measuredHeight must be numeric")
    result["errors"] = errors
    result["valid"] = len(errors) == 0
    return result

bad_height = {
    "type": "CityJSON",
    "version": "1.1",
    "transform": {"scale": [1, 1, 1], "translate": [0, 0, 0]},
    "CityObjects": {
        "BNEG": {
            "type": "Building",
            "attributes": {"measuredHeight": -5},
            "geometry": []
        }
    },
    "vertices": []
}
print(validate_cityjson_with_height(bad_height))

### 📖 Section 7 takeaway

- Validation should catch both structural issues and domain rules such as impossible negative heights.
- A lightweight custom validator is often enough for ETL screening before you hand data to heavier tooling.

---
## 📖 Section 8 — Reading and Writing CityJSON Files

Ways to read and write CityJSON in practice:
- Pure Python: `json.load()` / `json.dump()` works for any valid CityJSON file.
- `cjio`: a higher-level library for validation, upgrading, and CRS-aware workflows.
- GeoPandas: useful once you have converted 3D city objects into 2D analysis layers such as footprints.

**Best practice:** validate immediately after loading and always inspect the `transform` key before interpreting coordinates.

In [ ]:
# 💻 8.2  Write CityJSON to disk and read it back (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import json
from pathlib import Path

DATA_DIR = Path.home() / '.geospatial_course' / 'week09'
DATA_DIR.mkdir(parents=True, exist_ok=True)

out_path = DATA_DIR / 'sample_buildings.json'

# Write
with open(out_path, 'w', encoding='utf-8') as fh:
    json.dump(cj_full, fh, indent=2)

print(f"Written: {out_path}")
print(f"Size   : {out_path.stat().st_size:,} bytes")

# Read back
with open(out_path, 'r', encoding='utf-8') as fh:
    cj_loaded = json.load(fh)

# Validate loaded data
val = validate_cityjson(cj_loaded)
print(f"Loaded valid : {val['valid']}")
print(f"Objects      : {val['n_objects']}")
print(f"Vertices     : {val['n_vertices']}")

In [ ]:
# 💻 8.3  Optional cjio API demo (🏙️)
# ─────────────────────────────────────────────────────────────────────────────
demo_lines = [
    "# cjio high-level API (run if cjio is installed):",
    "",
    "import cjio.cityjson as cj_lib",
    "",
    "# Load",
    "cm = cj_lib.load('sample_buildings.json')",
    "",
    "# Metadata",
    "print(cm.get_version())",
    "print(cm.get_crs())",
    "",
    "# Iterate city objects",
    "for oid, obj in cm.get_cityobjects().items():",
    "    print(oid, obj.type, obj.attributes)",
    "",
    "# Validate",
    "errors = cm.validate()",
    "print(errors)",
    "",
    "# Reproject",
    "cm.reproject(4326)",
    "",
    "# Upgrade to latest version",
    "cm.upgrade_version('1.1')",
    "cm.save('sample_buildings_v11.json')",
]
print("
".join(demo_lines))

In [ ]:
# 💻 8.4  Inspect the loaded CityJSON object (🔷)
# ─────────────────────────────────────────────────────────────────────────────
print(sorted(cj_loaded.keys()))
first_id = next(iter(cj_loaded['CityObjects']))
print(f"First object id: {first_id}")
print(json.dumps(cj_loaded["CityObjects"][first_id], indent=2))

In [ ]:
# 💻 8.5  Check round-trip consistency (🔷)
# ─────────────────────────────────────────────────────────────────────────────
same_ids = sorted(cj_full['CityObjects']) == sorted(cj_loaded['CityObjects'])
same_vertex_count = len(cj_full['vertices']) == len(cj_loaded['vertices'])
same_transform = cj_full.get('transform') == cj_loaded.get('transform')
print(f"Same object ids    : {same_ids}")
print(f"Same vertex count  : {same_vertex_count}")
print(f"Same transform     : {same_transform}")

### 📖 Section 8 takeaway

- `json.load()` and `json.dump()` are often all you need for reliable CityJSON file I/O.
- Higher-level libraries such as `cjio` are most useful when you need validation, upgrading, or advanced management operations.

---
## 📖 Section 9 — Coordinate Reference Systems in CityJSON

CRS handling in CityJSON is explicit but easy to miss:
- `metadata.referenceSystem` often stores a URI such as `"https://www.opengis.net/def/crs/EPSG/0/28992"`.
- After decoding vertices, coordinates are already in the native CRS units, such as metres in RD New.
- For web maps or browser display, you will usually convert the extracted geometry to WGS84 (EPSG:4326).
- `geographicalExtent` stores `[minx, miny, minz, maxx, maxy, maxz]` in the native CRS.

In [ ]:
# 💻 9.2  Parse EPSG code and reproject footprints (🔷)
# ─────────────────────────────────────────────────────────────────────────────
from pyproj import Transformer
import re

def get_epsg(cj: dict) -> int:
    ref = cj.get('metadata', {}).get('referenceSystem', '')
    m = re.search(r'/(\d+)$', ref)
    return int(m.group(1)) if m else None

epsg = get_epsg(cj_full)
print(f"Detected EPSG: {epsg}")

# Reproject all footprints from RD New (28992) to WGS84 (4326)
transformer = Transformer.from_crs(epsg or 28992, 4326, always_xy=True)

from shapely.ops import transform as shp_transform

gdf_wgs = gdf.copy()
gdf_wgs['geometry'] = gdf.geometry.apply(
    lambda g: shp_transform(transformer.transform, g)
)
gdf_wgs = gdf_wgs.set_crs('EPSG:4326', allow_override=True)

print(f"\nReprojected footprints (WGS84):")
for _, row in gdf_wgs.iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    print(f"  {row['id']}: centroid=({cx:.6f}, {cy:.6f})  height={row['measuredHeight']} m")

In [ ]:
# 💻 9.3  Save reprojected footprints to GeoJSON (🔷)
# ─────────────────────────────────────────────────────────────────────────────
geojson_path = DATA_DIR / 'building_footprints_wgs84.geojson'
gdf_wgs.to_file(str(geojson_path), driver='GeoJSON')
print(f"GeoJSON saved: {geojson_path}")
print(f"Size         : {geojson_path.stat().st_size:,} bytes")

# Preview the GeoJSON structure
with open(geojson_path) as f:
    gj = json.load(f)
print(f"Features     : {len(gj['features'])}")
print(f"First feature properties: {gj['features'][0]['properties']}")

In [ ]:
# 💻 9.4  Reproject the geographical extent corners (🔷)
# ─────────────────────────────────────────────────────────────────────────────
extent = cj_full['metadata']['geographicalExtent']
minx, miny, minz, maxx, maxy, maxz = extent
ll = transformer.transform(minx, miny)
ur = transformer.transform(maxx, maxy)
print(f"Native extent : {extent}")
print(f"WGS84 extent  : ({ll[0]:.6f}, {ll[1]:.6f}) → ({ur[0]:.6f}, {ur[1]:.6f})")

In [ ]:
# 💻 9.5  Quick WGS84 preview map (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

ax = gdf_wgs.plot(column="measuredHeight", cmap="viridis", legend=True, figsize=(6, 5), edgecolor="black")
ax.set_title("Building footprints in WGS84")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

### 📖 Section 9 takeaway

- Always decode first, then reproject: CRS logic applies to the **real** coordinates, not the stored integers.
- Parsing the EPSG code from `metadata.referenceSystem` lets you connect CityJSON to standard GIS workflows.

---
## 📖 Section 10 / Mini-Lab — Extract Building Footprints and Metadata

Scenario: You are a 3D city data analyst. You have received a synthetic CityJSON dataset of 6 buildings in Amsterdam. Your task is to parse the dataset, validate it, extract footprints and heights, reproject to WGS84, compute statistics, and export a clean GeoJSON report.

In [ ]:
# 🔬 Step 1 — Build the dataset
# ─────────────────────────────────────────────────────────────────────────────
lab_spec = [
    ("LAB_A", 84720.0, 446820.0, 16.0, 12.0,  6.0, 1910, "residential"),
    ("LAB_B", 84745.0, 446820.0, 20.0, 14.0, 18.0, 1965, "office"),
    ("LAB_C", 84775.0, 446820.0, 22.0, 16.0, 42.0, 2018, "commercial"),
    ("LAB_D", 84725.0, 446850.0, 12.0, 10.0,  5.0, 1905, "industrial"),
    ("LAB_E", 84750.0, 446850.0, 18.0, 18.0, 27.0, 1999, "residential"),
    ("LAB_F", 84780.0, 446850.0, 24.0, 18.0, 50.0, 2020, "office"),
]

def apply_offset(boundaries, offset):
    def _shift(item):
        if isinstance(item[0], int):
            return [v + offset for v in item]
        return [_shift(sub) for sub in item]
    return [_shift(shell) for shell in boundaries]

lab_vertices = []
lab_objects = {}
for bid, cx, cy, w, d, h, year_built, func in lab_spec:
    offset = len(lab_vertices)
    verts, geom = make_box_building(cx, cy, w, d, h, tf)
    lab_vertices.extend(verts)
    geom['boundaries'] = apply_offset(geom['boundaries'], offset)
    lab_objects[bid] = {
        "type": "Building",
        "attributes": {
            "measuredHeight": h,
            "yearBuilt": year_built,
            "function": func,
        },
        "geometry": [geom],
    }

lab_cj = {
    "type": "CityJSON",
    "version": "1.1",
    "transform": tf,
    "metadata": {
        "referenceSystem": "https://www.opengis.net/def/crs/EPSG/0/28992",
        "geographicalExtent": [84720.0, 446820.0, 0.0, 84804.0, 446868.0, 50.0],
    },
    "CityObjects": lab_objects,
    "vertices": lab_vertices,
}

lab_summary = pd.DataFrame([
    {
        "id": bid,
        "function": obj["attributes"]["function"],
        "height_m": obj["attributes"]["measuredHeight"],
        "year_built": obj["attributes"]["yearBuilt"],
    }
    for bid, obj in lab_cj['CityObjects'].items()
])
print(f"Objects  : {len(lab_cj['CityObjects'])}")
print(f"Vertices : {len(lab_cj['vertices'])}")
print(lab_summary.to_string(index=False))

In [ ]:
# 🔬 Step 2 — Validate the dataset
# ─────────────────────────────────────────────────────────────────────────────
lab_validation = validate_cityjson(lab_cj)
print(f"Valid    : {lab_validation['valid']}")
print(f"Errors   : {lab_validation['errors']}")
print(f"Warnings : {lab_validation['warnings']}")
assert len(lab_validation["errors"]) == 0, "Expected 0 validation errors"
print("✅ Validation passed with 0 errors")

In [ ]:
# 🔬 Step 3 — Extract footprints and compare heights
# ─────────────────────────────────────────────────────────────────────────────
lab_real = decode_vertices(lab_cj['vertices'], lab_cj['transform'])
lab_rows = []
for bid, obj in lab_cj['CityObjects'].items():
    fp = next((extract_footprint(g, lab_real) for g in obj["geometry"]), None)
    geom_h = get_height_from_geometry(obj, lab_real)
    attr_h = obj["attributes"].get("measuredHeight")
    lab_rows.append({
        "id": bid,
        "function": obj["attributes"]["function"],
        "year_built": obj["attributes"]["yearBuilt"],
        "measuredHeight": attr_h,
        "geom_height": round(float(geom_h), 2) if geom_h is not None else None,
        "height_diff_m": round(abs(attr_h - geom_h), 3) if geom_h is not None else None,
        "geometry": fp,
    })

lab_gdf = gpd.GeoDataFrame(lab_rows, geometry='geometry', crs='EPSG:28992')
print(lab_gdf[["id", "function", "measuredHeight", "geom_height", "height_diff_m"]].to_string(index=False))

In [ ]:
# 🔬 Step 4 — Compute statistics
# ─────────────────────────────────────────────────────────────────────────────
lab_gdf['footprint_m2'] = lab_gdf.geometry.area.round(2)
lab_gdf['volume_m3'] = (lab_gdf['footprint_m2'] * lab_gdf['measuredHeight']).round(2)
print(f"Mean height           : {lab_gdf['measuredHeight'].mean():.2f} m")
print(f"Min height            : {lab_gdf['measuredHeight'].min():.2f} m")
print(f"Max height            : {lab_gdf['measuredHeight'].max():.2f} m")
print(f"Total footprint area  : {lab_gdf['footprint_m2'].sum():.2f} m²")
print(f"Total estimated volume: {lab_gdf['volume_m3'].sum():.2f} m³")
print("
Count by function:")
print(lab_gdf.groupby("function").size().to_string())

In [ ]:
# 🔬 Step 5 — Reproject and map
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

lab_gdf_wgs = lab_gdf.to_crs(4326)
ax = lab_gdf.plot(column="measuredHeight", cmap="plasma", legend=True, figsize=(7, 5), edgecolor="black")
ax.set_title("Lab building footprints coloured by height")
ax.set_xlabel("RD New X")
ax.set_ylabel("RD New Y")
plt.show()
print(lab_gdf_wgs[["id", "function", "measuredHeight"]].to_string(index=False))

In [ ]:
# 🔬 Step 6 — Export to GeoJSON
# ─────────────────────────────────────────────────────────────────────────────
lab_geojson_path = DATA_DIR / 'week09_lab_buildings.geojson'
lab_gdf_wgs.to_file(str(lab_geojson_path), driver="GeoJSON")
print(f"GeoJSON saved: {lab_geojson_path}")
with open(lab_geojson_path, "r", encoding="utf-8") as fh:
    lab_geojson = json.load(fh)
print(f"First feature properties: {lab_geojson['features'][0]['properties']}")

### 📖 Section 10 takeaway

- A practical CityJSON workflow is: **parse → validate → extract geometry → compute metrics → reproject → export**.
- This pattern is exactly what you will reuse in a capstone-scale ETL pipeline.

## 🏁 Week 9 Summary

| Section | Key Concept |
|---------|-------------|
| 1 | CityJSON schema: type, version, CityObjects, vertices, transform |
| 2 | Integer vertex compression: real = int * scale + translate |
| 3 | Object types: Building, Road, WaterBody + parent/child hierarchy |
| 4 | Geometry types: MultiSurface, Solid, boundaries nesting, LoD |
| 5 | Footprint extraction: find lowest face ring → Shapely Polygon |
| 6 | Height extraction: measuredHeight attr vs geometry-derived |
| 7 | Validation: missing keys, out-of-range indices, empty geometry |
| 8 | File I/O: json.load/dump + cjio for high-level operations |
| 9 | CRS handling: parse EPSG from referenceSystem, reproject with PyProj |
| 10 | Mini-Lab: parse → validate → extract → stats → reproject → export |

### ☑️ Self-assessment checklist
- [ ] I can explain the CityJSON transform model (integer compression)
- [ ] I can decode vertices using scale + translate
- [ ] I know the common CityJSON object types and their hierarchy
- [ ] I can traverse Solid/MultiSurface boundary arrays recursively
- [ ] I can extract a 2D footprint polygon from a LoD1 Solid geometry
- [ ] I can derive building height from both attributes and geometry
- [ ] I can validate a CityJSON file with custom Python checks
- [ ] I can read and write CityJSON with json.load/dump
- [ ] I can extract the EPSG code from metadata.referenceSystem
- [ ] I am ready to complete the Week 10 Capstone project

## 📚 Week 10 Preview

| Topic | What you'll learn |
|-------|-------------------|
| Capstone project design | Combining files, Python, PostGIS, and CityJSON |
| ETL pipeline | Full data ingest from CSV + CityJSON to PostGIS |
| Spatial analysis | Applying Weeks 5–8 skills on real data |
| Presentation | Notebook structure, narrative, and reproducibility |
| Deliverables | Notebook + scripts + SQL + short presentation |

## 📖 Further Reading

| Resource | Link |
|----------|------|
| CityJSON spec | https://www.cityjson.org/specs/ |
| CityJSON examples | https://www.cityjson.org/datasets/ |
| cjio library | https://github.com/cityjson/cjio |
| CityGML standard | https://www.ogc.org/standard/citygml/ |
| 3D BAG (Netherlands) | https://3dbag.nl/en/download |

---
*Next: **Week 10 — Capstone Project** — design, implement, and present a complete geospatial solution combining Python, PostGIS, and CityJSON.*